In [1]:
%matplotlib widget

import os
import numpy as np
from PIL import Image
import cv2
import numpy as np
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import patheffects as pe
import pytesseract
import matplotlib.patches as mpatches
from matplotlib.patches import Rectangle, Polygon
import image_utils
from get_cards_tablet import card_peek_width, card_width, get_cards, card_height
import cv2
import numpy as np
import os
import matplotlib.pyplot as plt
from image_utils import load_rdb, save_rgb_png
from android_capture import ScreenCapture
import time
import image_utils
import ocr_cards
import matplotlib.pyplot as plt
from android_capture import ScreenCapture
import time
import image_utils
import get_cards_tablet
import ocr_cards
import matplotlib.pyplot as plt
from android_actions import AndroidBJTabletActor
from blackjack.actions import PlayerAction
from blackjack.cards import Card, Rank
from blackjack.hand import ValueOnlyHand
from blackjack.blackjack_round import BJRound, BJStage
from blackjack.rules import BJRules
from strategy import CardCounter
from blackjack.actions import PlayerAction, DealerAction

In [2]:
rules = BJRules(
    dealer_checks_blackjack = True,
    dealer_hits_soft_17 = False,
    allow_late_surrender = False,
    allow_early_surrender_on_ten = False,
    allow_early_surrender_on_ace = False,
    allow_early_surrender_on_all = False,
    dealer_shows_card_on_surrender = True,
    allow_insurance_vs_ace = True,
    natural_blackjack_payout = 3 / 2,
    surrender_payout = 1 / 2,
    insurance_payout = 2 / 1,
    max_splits_allowed = 1,
    allow_action_on_split_aces = True,
    allow_double_after_split = True,
    allow_double_on_soft = True,
    allow_split_different_tens = True,
    no_natural_bj_on_split = True,
    split_order_reversed = True
)

In [ ]:
def card_list_str_to_rank_values(card_list_str):
    return tuple(Rank(s[:-1]).rank_value() for s in card_list_str)


class Player:
    def __init__(self, scr_taker, card_counter):
        self.actor = AndroidBJTabletActor()
        self.scr_taker = scr_taker  # "/media/maxim/T7/frames/")
        self.round = BJRound(rules)
        self.active_hand_position = get_cards_tablet.HandPosition.NONE
        self.card_counter = card_counter

    def is_pre_shuffle(self):
        game_img = self.scr_taker.get_screen()
        return get_cards_tablet.is_pre_shuffle(game_img)

    def reset_hands(self):
        self.round = BJRound(rules)
        self.active_hand_position = get_cards_tablet.HandPosition.NONE

    def play_round(self, bet=1000):
        self.reset_hands()

        game_img = self.scr_taker.get_screen()
        while not get_cards_tablet.is_table_empty(game_img):
            time.sleep(0.5)
            game_img = self.scr_taker.get_screen()
        self.round.start_round(bet)
        self.actor.place_bet(bet)
        self.actor.deal()
        self.wait_for_initial_cards()

        print("initial hand")
        print(self.round)

        print("true count: ", self.card_counter.get_true_count())

        player_had_bj = self.round.player_hands[0].is_natural_blackjack()

        if not player_had_bj:
            if self.round.get_dealer_hand()[0] == 10:
                self.wait_for_dealer_bj_or_action_request()
            elif self.round.get_dealer_hand()[0] == 11:
                # if dealer has A, insurance is always offered
                self.refuse_insurance()
                self.wait_for_dealer_bj_or_action_request()
            else:
                self.wait_for_action_request()
        else:  # player has blackjack
            if self.round.get_dealer_hand()[0] == 11:
                self.refuse_insurance()
            # sometimes dealer both 2 cards are captured by wait_for_initial_cards
            # instead of one
            self.wait_for_dealer_full_hand(only_two_cards=True)

        dealer_has_bj = self.round.get_dealer_hand().is_natural_blackjack()
        if player_had_bj or dealer_has_bj:
            self.end_round()
            return
        
        if PlayerAction.SPLIT in self.round.get_available_actions():
            self.split()

        while self.round.need_player_action():
            time.sleep(1)
            hand = self.round.get_active_player_hand()
            if len(hand) == 2 and hand.get_best_value() in [9, 10, 11]:
                self.double()
            else:
                while not hand.is_bust() and not hand.get_best_value() == 21: #  and hand.get_best_value() < 17: 
                    self.hit()  # hand is a pointer to a mutable object so should be updated
                player_bust = hand.is_bust()
                player_21 = hand.get_best_value() == 21
                if not player_bust and not player_21:
                    self.stand()
        # when player is bust dealer shows only one extra card
        self.wait_for_dealer_full_hand(
            only_two_cards=all(hand.is_bust() for hand in self.round.player_hands)
        )
        self.end_round()

    def double(self):
        self.wait_for_action_btn()
        self.actor.double()
        self.round.take_action(PlayerAction.DOUBLE)
        self.wait_for_player_card()

    def split(self):
        self.wait_for_action_btn()
        self.actor.split()
        self.round.take_action(PlayerAction.SPLIT)
        self.wait_for_player_card()
        self.wait_for_player_card()

    def hit(self):
        self.wait_for_action_btn()
        self.actor.hit()
        self.round.take_action(PlayerAction.HIT)
        self.wait_for_player_card()

    def stand(self):
        self.wait_for_action_btn()
        self.actor.stand()
        self.round.take_action(PlayerAction.STAND)

    def refuse_insurance(self):
        self.wait_for_insurance_option()
        self.actor.refuse_insurance()
        self.round.take_action(PlayerAction.REFUSE_INSURANCE)

    def take_insurance(self):
        self.wait_for_insurance_option()
        self.actor.take_insurance()
        self.round.take_action(PlayerAction.TAKE_INSURANCE)

    def take_card(self, card):
        self.round.take_card(card)
        self.card_counter.update_count(card)

    def wait_for_action_btn(self):
        while True:
            game_img = self.scr_taker.get_screen()
            can_stand = get_cards_tablet.can_hit_stand(game_img)
            if can_stand:
                break

    def end_round(self):
        print("final stage")
        assert self.round.get_stage() == BJStage.ROUND_OVER
        print(self.round)

        self.actor.rebuy()
        

    def wait_for_initial_cards(self):
        print("wait_for_initial_cards")
        game_img = self.scr_taker.get_screen()
        dealer_cards_new = card_list_str_to_rank_values(get_cards_tablet.get_dealer_cards(game_img))
        middle_cards_new = card_list_str_to_rank_values(get_cards_tablet.get_player_cards_middle(game_img))

        dealer_cards_set = set([dealer_cards_new])
        middle_cards_set = set([dealer_cards_new])

        cards_confirmed = False
        while True:
            if (
                cards_confirmed 
                and len(dealer_cards_new) >= 1
                and len(middle_cards_new) >= 2
            ):
                break

            game_img = self.scr_taker.get_screen()
            dealer_cards_new = card_list_str_to_rank_values(get_cards_tablet.get_dealer_cards(game_img))
            middle_cards_new = card_list_str_to_rank_values(get_cards_tablet.get_player_cards_middle(game_img))

            if dealer_cards_new in dealer_cards_set and middle_cards_new in middle_cards_set:
                cards_confirmed = True
            else:
                cards_confirmed = False
                dealer_cards_set.add(dealer_cards_new)
                middle_cards_set.add(middle_cards_new)

        self.take_card(middle_cards_new[0])
        self.take_card(middle_cards_new[1])
        self.take_card(dealer_cards_new[0])
        # on this stage the dealer can show 2 cards - if she has blackjack 
        # or if player has bj and she shows that she doesn't have one
        if len(dealer_cards_new) >= 2:
            if sum(dealer_cards_new) == 21:
                self.round.take_action(DealerAction.CONFIRM_BLACKJACK)
            else:
                self.round.take_action(DealerAction.CONFIRM_NO_BLACKJACK)
            self.take_card(dealer_cards_new[1])


    def wait_for_insurance_option(self):
        print("wait_for_insurance_option")
        game_img = self.scr_taker.get_screen()
        while not get_cards_tablet.is_insurance_offered(game_img):
            game_img = self.scr_taker.get_screen()


    def wait_for_action_request(self):
        print("wait_for_action_request")
        while True:
            if self.active_hand_position != get_cards_tablet.HandPosition.NONE:
                break
            game_img = self.scr_taker.get_screen()
            self.active_hand_position = get_cards_tablet.get_active_hand(game_img)


    def wait_for_dealer_bj_or_action_request(self):
        # after insurance refused dealer either shows bj or points to my hand
        print("wait_for_dealer_bj_or_action_request")
        game_img = self.scr_taker.get_screen()
        dealer_cards_new = card_list_str_to_rank_values(get_cards_tablet.get_dealer_cards(game_img))
        dealer_cards_set = set([dealer_cards_new])
        
        dealer_cards_confirmed = False
        action_request = False
        dealer_blackjack = False
        while True:
            # dealer shows blackjack
            if dealer_cards_confirmed and len(dealer_cards_new) >= 2:
                dealer_blackjack = True
                break
            # dealer asks for action
            if self.active_hand_position != get_cards_tablet.HandPosition.NONE:
                action_request = True
                break
                
            game_img = self.scr_taker.get_screen()
            self.active_hand_position = get_cards_tablet.get_active_hand(game_img)
            dealer_cards_new = card_list_str_to_rank_values(get_cards_tablet.get_dealer_cards(game_img))
            
            if dealer_cards_new in dealer_cards_set:
                dealer_cards_confirmed = True
            else:
                dealer_cards_set.add(dealer_cards_new)
                dealer_cards_confirmed = False

        if dealer_blackjack:
            self.round.take_action(DealerAction.CONFIRM_BLACKJACK)
            self.take_card(dealer_cards_new[1])
        else:
            self.round.take_action(DealerAction.CONFIRM_NO_BLACKJACK)
        

    def wait_for_dealer_full_hand(self, only_two_cards=False):
        print("wait_for_dealer_full_hand")
        if only_two_cards and len(self.round.get_dealer_hand()) >= 2:
            return
        
        game_img = self.scr_taker.get_screen()

        dealer_cards_new = card_list_str_to_rank_values(get_cards_tablet.get_dealer_cards(game_img))
        dealer_cards_set = set([dealer_cards_new])
        current_hand = ValueOnlyHand(dealer_cards_new)
        
        dealer_confirmed = False
        
        n_start_cards = len(self.round.get_dealer_hand())
        while True:
            if dealer_confirmed: 
                if only_two_cards:
                    if len(current_hand) >= 2:
                        break
                else:
                    if current_hand.is_bust() or current_hand.get_best_value() >= 17:
                        break
        
            game_img = self.scr_taker.get_screen()
            dealer_cards_new = card_list_str_to_rank_values(get_cards_tablet.get_dealer_cards(game_img))
            print(dealer_cards_new)
            if dealer_cards_new in dealer_cards_set:
                dealer_confirmed = True
                current_hand = ValueOnlyHand(dealer_cards_new)
            else:
                dealer_confirmed = False
                dealer_cards_set.add(dealer_cards_new)
                current_hand = None

        if self.round.get_stage() == BJStage.DEALER_CHECK_BJ:
            print("second check bj tested")
            if sum(dealer_cards_new) == 21:
                self.round.take_action(DealerAction.CONFIRM_BLACKJACK)
            else:
                self.round.take_action(DealerAction.CONFIRM_NO_BLACKJACK)

        for c in current_hand.cards[n_start_cards:]:
            self.take_card(c)


    def wait_for_player_card(self):
        print("wait_for_player_card")
        round_hand_idx = self.round.active_hand_idx
        n_hands = len(self.round.player_hands)
        if n_hands == 1 and round_hand_idx == 0:
            self.active_hand_position = get_cards_tablet.HandPosition.MIDDLE
        elif n_hands == 2 and round_hand_idx == 0:
            # hands go in the order of right -> left
            self.active_hand_position = get_cards_tablet.HandPosition.RIGHT
        elif n_hands == 2 and round_hand_idx == 1:
            self.active_hand_position = get_cards_tablet.HandPosition.LEFT
        else:
            raise ValueError(f"Unsupported hands state n={n_hands}, idx={round_hand_idx}")
        
        hand = self.round.get_active_player_hand()
        
        # 4 cards in each line 
        n_cards_last_line = len(hand) % 4
        if len(hand) > 0 and n_cards_last_line == 0:
            n_cards_last_line = 4

        print(n_cards_last_line)
        # 6th card covers cards 1,2,3,4 (entire previous line)
        visible_tail_idx = slice(-n_cards_last_line, None)
        if n_cards_last_line == 4:
            # 5th card covers cards 1,2,3
            visible_tail_idx = slice(-1, None)
        # 6th (2nd in the new line) will cover the entire previous line
        # no need to adjust slice, just take the last (current) line
        
        visible_old_tail = tuple(hand[visible_tail_idx])
        game_img = self.scr_taker.get_screen()
        hand_cards_new = card_list_str_to_rank_values(
            get_cards_tablet.get_player_cards(game_img, self.active_hand_position)
        )
        hand_cards_set = set([hand_cards_new])
        
        hand_confirmed = False
        while True:
            print(hand_confirmed, hand_cards_new, visible_old_tail)
            if (
                hand_confirmed 
                and hand_cards_new[:-1] == visible_old_tail
                and len(hand_cards_new) == len(visible_old_tail) + 1
            ):
                break

            game_img = self.scr_taker.get_screen()
            hand_cards_new = card_list_str_to_rank_values(
                get_cards_tablet.get_player_cards(game_img, self.active_hand_position)
            )
            
            if hand_cards_new in hand_cards_set:
                hand_confirmed = True
            else:
                hand_confirmed = False
                hand_cards_set.add(hand_cards_new)
        
        new_card = hand_cards_new[-1]
        self.take_card(new_card)



In [4]:
src_taker = None
del src_taker

all_cards = []
net_value = []

src_taker = ScreenCapture()
game_img = src_taker.get_screen()
initial_penetration = 0 # get_cards_tablet.get_shoe_penetration(game_img)
print("initial_penetration", initial_penetration)
counter = CardCounter(n_decks=6, current_penetration=initial_penetration)

initial_penetration 0


In [5]:
actor = AndroidBJTabletActor()

actor.leave_table()

In [6]:
game_img = src_taker.get_screen()
while not get_cards_tablet.can_create_private_table(game_img):
    time.sleep(1)
    game_img = src_taker.get_screen()

while get_cards_tablet.can_create_private_table(game_img):
    actor.create_private_table()
    game_img = src_taker.get_screen()

In [7]:
p = Player(src_taker, counter)
game_img = src_taker.get_screen()

while not get_cards_tablet.is_pre_shuffle(game_img):
    print(f"start, tc = {counter.get_true_count()}")
    
    p.play_round()

    for hand in p.round.player_hands:
        all_cards.extend(hand.cards)
    all_cards.extend(p.round.dealer_hand.cards)
    
    net_value.append(p.round.get_player_value())
    
    game_img = src_taker.get_screen()

    time.sleep(1)

start, tc = 0.0
wait_for_initial_cards
initial hand
Last card: 4
Dealer 4,X
Player 5,10(15)[$1000]
true count:  0.16828478964401294
wait_for_action_request
wait_for_player_card
2
False (5, 10) (5, 10)
True (5, 10) (5, 10)
True (5, 10) (5, 10)
True (5, 10) (5, 10)
True (5, 10) (5, 10)
True (5, 10) (5, 10)
True (5, 10) (5, 10)
True (5, 10) (5, 10)
True (5, 10) (5, 10)
True (5, 10) (5, 10)
True (5, 10) (5, 10)
True (5, 10) (5, 10)
False (5, 5) (5, 10)
False (5, 10, 9) (5, 10)
True (5, 10, 9) (5, 10)
wait_for_dealer_full_hand
(4,)
(4,)
(4,)
(4,)
(4,)
(4,)
(4,)
(4,)
(4,)
(4,)
(4,)
(4,)
(4,)
(4,)
(4,)
(4,)
(4,)
(4,)
(4,)
(4,)
(4,)
(4,)
(4,)
(4,)
(4,)
(4,)
(4,)
(4,)
(4,)
(4,)
(4,)
(4,)
(4,)
(4,)
()
(4,)
(4,)
(4,)
(4,)
(4,)
(4,)
()
(4,)
(4,)
(4,)
(4,)
(4,)
(4,)
(4,)
(4,)
(4,)
(4,)
(4,)
(4,)
(4,)
(4,)
(4,)
(4,)
(4,)
(4,)
(4,)
(4,)
(4,)
(4,)
(4,)
(4,)
(4,)
(4,)
(4,)
()
()
()
()
()
()
()
()
()
()
()
()
()
()
()
()
()
()
()
()
()
()
()
()
()
()
()
()
()
()
()
()
()
()
()
(4,)
(4, 4)
(4, 4)
final s

KeyboardInterrupt: 

In [ ]:
print(p.round)

Last card: 9
Dealer 2,X
Player 2,10,9(21)-stand[$1000]


In [ ]:
print(p.round)

NameError: name 'p' is not defined

In [ ]:
p.round.get_stage()

NameError: name 'p' is not defined

In [ ]:
from collections import Counter

In [ ]:
card_count = Counter()
for card in cards:
    card_count[str(card)] += 1

In [ ]:
card_count

Counter({'10': 1, '7': 1, '9': 1, '8': 1})

In [ ]:
len(cards) / 52

0.07692307692307693

In [ ]:
len(cards) / 52 / 6 * 100

1.2820512820512822

In [ ]:
# conclusion is that there are 6 decks
# deck penetration is 82.37179487179488